In [ ]:
## Install necessary dependencies
!pip install -q unsloth bitsandbytes torch accelerate xformers transformers datasets ollama

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
def fine_tune_model(model_name, task, dataset_name):
    print(f"Fine-tuning {model_name} for {task}")

    try:
        # Load dataset with error handling and streaming mode for large datasets
        dataset = load_dataset(dataset_name, streaming=True)
        print("Dataset loaded successfully.")
    except Exception as e:
        print(f"Dataset loading error: {e}")
        return

    # Convert dataset to iterable if streaming is enabled
    if dataset and isinstance(dataset, dict):
        dataset = dataset.map(lambda x: x, remove_columns=[])

    # Load model using Unsloth
    # Try loading in 4bit to reduce memory footprint
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name,
        load_in_4bit=True,  # Enable 4bit quantization
        load_in_8bit=False, # Disable 8bit quantization
        device_map="cuda:0", # or "cpu" if you want to load on CPU
    )

    # Apply LoRA adapters
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing=True # Already enabled, but ensure it's working
    )

    # Training arguments - Reduced batch size and increased gradient accumulation
    training_args = TrainingArguments(
        output_dir=f"./{model_name}_{task}",
        per_device_train_batch_size=1,  # Reduced batch size
        per_device_eval_batch_size=1,  # Reduced batch size
        gradient_accumulation_steps=8,  # Increased gradient accumulation
        logging_steps=10,
        eval_strategy="epoch" if "test" in dataset else "no",
        save_strategy="epoch",
        learning_rate=2e-4,
        num_train_epochs=1,
        weight_decay=0.01,
        push_to_hub=False,
        fp16=True,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"] if "train" in dataset else dataset,
        eval_dataset=dataset["test"] if "test" in dataset else None,
    )

    # Train the model
    trainer.train()

    # Save and evaluate
    trainer.save_model(f"./{model_name}_{task}")
    tokenizer.save_pretrained(f"./{model_name}_{task}")

    print(f"{model_name} fine-tuned for {task} and saved.")

In [ ]:
# Define datasets for each task
datasets = {
    "code_generation": "codeparrot/github-code",
    "chatbot": "openassistant/oasst1",
    "sentiment_classification": "imdb",
    "summarization": "cnn_dailymail"
}

In [ ]:
# Run fine-tuning for all models
for model, task in models_tasks.items():
    fine_tune_model(model, task, datasets[task])

Fine-tuning meta-llama/Meta-Llama-3-8B for code_generation
Dataset loaded successfully.
==((====))==  Unsloth 2025.3.17: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

In [ ]:
# Export fine-tuned model to Ollama
!huggingface-cli download your_username/model_lora_adapter --local-dir /content/model_lora_adapter


In [ ]:
# Convert to GGML for Ollama
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && pip install -r requirements.txt
!python llama.cpp/convert-lora-to-ggml.py /content/model_lora_adapter/adapter_config.json

In [ ]:
# Create Ollama Modelfile
modelfile_content = '''
FROM tinyllama:latest
ADAPTER ./ggml-adapter-model.bin
TEMPLATE """{{ .System }}
>>> User: {{ .Prompt }}
>>> Assistant:
"""
PARAMETER stop ">>> User:"
PARAMETER stop ">>> Assistant:"
'''
with open("ModelfileTinyllama", "w") as f:
    f.write(modelfile_content)

# Create and run the model in Ollama
!ollama create tinyadap -f ./ModelfileTinyllama
!ollama run tinyadap